In [6]:
# 第一部分：导入必要的库
# ============================================

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA          # ← 新增
from sklearn.model_selection import train_test_split
import mindspore as ms
from mindspore import nn, Tensor
from mindspore.nn import SoftmaxCrossEntropyWithLogits
from mindspore.common.initializer import Normal
import mindspore.dataset as ds


from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')


In [7]:
# 第二部分：可调参数配置区（PCA降维 + MLP 经典对照组）
# ============================================

# 数据相关参数（与量子实验保持一致）
SAMPLE_RATIO = 1            # 数据采样比例
TEST_SIZE = 0.2             # 测试集比例
PCA_COMPONENTS = 8          # 降维目标维度（与量子比特数一致）
BATCH_SIZE = 32             # 批处理大小

# 降维方法
DIM_REDUCTION_METHOD = 'pca'
# MLP 参数
MLP_HIDDEN_DIMS = [64, 32]  # 两个隐藏层维度（输入8 -> 64 -> 32 -> 10）

# 训练参数（沿用量子实验）
NUM_EPOCHS = 150             # 训练轮数
LEARNING_RATE = 0.01         # 学习率

# MindSpore 配置
MS_MODE = ms.PYNATIVE_MODE
MS_DEVICE = "CPU"

# 随机种子（与量子实验一致）
SEED = 42

# 初始化设置
# ============================================
ms.set_context(mode=MS_MODE, device_target=MS_DEVICE)
ms.set_seed(SEED)
np.random.seed(SEED)


[WARNING] ME(43296:29248,MainProcess):2026-10-07-01:51:27.292.000 [mindspore\context.py:1412] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


In [8]:
# 第三部分：数据加载与PCA降维预处理（与量子实验一致）
# ============================================

import os
import gzip
import struct
import urllib.request

_MNIST_FILES = {
    'train_images': ('train-images-idx3-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz'),
    'train_labels': ('train-labels-idx1-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-labels-idx1-ubyte.gz'),
    'test_images':  ('t10k-images-idx3-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-images-idx3-ubyte.gz'),
    'test_labels':  ('t10k-labels-idx1-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz'),
}

def _ensure_mnist_files(data_dir='./MNIST'):
    """下载/校验 MNIST 原始 IDX 文件（有本地缓存则不重复下载）"""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, (fname, url) in _MNIST_FILES.items():
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            print(f"下载 {fname} ...")
            try:
                urllib.request.urlretrieve(url, path)
            except Exception as e:
                raise RuntimeError(
                    f"下载 {fname} 失败：{e}。请确认本机可联网，"
                    f"或把原始电脑上的 ./MNIST 目录（含 4 个 .gz 文件）拷贝到本机后重试。"
                )
        paths[key] = path
    return paths

def _read_idx_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols)

def _read_idx_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

def load_mnist_data():
    """加载MNIST数据集（不依赖 torchvision，直接解析官方 IDX 文件，数据与 torchvision 一致）"""
    print("正在加载MNIST数据集...")
    paths = _ensure_mnist_files('./MNIST')

    train_images = _read_idx_images(paths['train_images'])   # (60000, 28, 28)
    train_labels = _read_idx_labels(paths['train_labels'])   # (60000,)
    test_images  = _read_idx_images(paths['test_images'])    # (10000, 28, 28)
    test_labels  = _read_idx_labels(paths['test_labels'])    # (10000,)

    def flatten(imgs):
        return imgs.astype(np.float32).reshape(len(imgs), -1) / 255.0

    # 合并（顺序与 torchvision 一致：先 60000 训练 + 后 10000 测试）
    all_images = np.concatenate([flatten(train_images), flatten(test_images)])
    all_labels = np.concatenate([train_labels, test_labels]).astype(np.int64)

    sampled_indices = np.random.choice(
        len(all_images),
        int(len(all_images) * SAMPLE_RATIO),
        replace=False
    )
    sampled_images = all_images[sampled_indices]
    sampled_labels = all_labels[sampled_indices]

    print(f"总样本数: {len(all_images)}, 采样后: {len(sampled_images)}")
    print(f"类别分布: {np.unique(sampled_labels, return_counts=True)}")

    return sampled_images, sampled_labels

def preprocess_data(images, labels, pca=None, scaler=None, is_train=True):
    """
    数据预处理：PCA降维 + MinMax归一化到[-1,1]（与量子实验一致）
    训练集 fit，测试集 transform；返回 (features, pca, scaler)
    """
    if is_train:
        pca = PCA(n_components=PCA_COMPONENTS)
        pca_data = pca.fit_transform(images)
        print(f"PCA解释方差比: {np.sum(pca.explained_variance_ratio_):.4f}")

        scaler = MinMaxScaler(feature_range=(-1, 1))
        scaled_data = scaler.fit_transform(pca_data)
    else:
        pca_data = pca.transform(images)
        scaled_data = scaler.transform(pca_data)

    return scaled_data, pca, scaler


In [9]:
# 第四部分：MLP 分类器定义与训练（经典对照组，替换量子电路 + 线性头）
# ============================================

class MLPClassifier(nn.Cell):
    """经典全连接网络分类器（两个隐藏层，权重初始化与量子模型分类头一致 Normal(0.02)）"""
    def __init__(self, input_dim=8, hidden_dims=[64, 32], num_classes=10):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Dense(input_dim, hidden_dims[0], weight_init=Normal(0.02))
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Dense(hidden_dims[0], hidden_dims[1], weight_init=Normal(0.02))
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Dense(hidden_dims[1], num_classes, weight_init=Normal(0.02))

    def construct(self, x):
        if len(x.shape) == 1:
            x = x.reshape(1, -1)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

def train_mlp(train_dataset, val_dataset, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE):
    """训练MLP，返回 (训练损失曲线, 验证准确率曲线, 最佳验证准确率, 训练集准确率)"""
    print(f"\n开始训练 MLP (结构: {PCA_COMPONENTS} -> {MLP_HIDDEN_DIMS[0]} -> {MLP_HIDDEN_DIMS[1]} -> 10)")

    net = MLPClassifier(input_dim=PCA_COMPONENTS, hidden_dims=MLP_HIDDEN_DIMS, num_classes=10)
    loss_fn = SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
    optimizer = nn.Adam(net.trainable_params(), learning_rate=learning_rate)

    def forward_fn(data, label):
        if len(data.shape) == 1:
            data = data.reshape(1, -1)
        logits = net(data)
        loss = loss_fn(logits, label)
        return loss, logits

    grad_fn = ms.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

    train_losses = []
    val_accuracies = []
    best_val_acc = 0.0

    print(f"\n开始训练循环 ({num_epochs} epochs)...")
    for epoch in range(num_epochs):
        net.set_train()
        epoch_loss = 0
        step = 0
        for data, label in train_dataset.create_tuple_iterator():
            (loss, _), grads = grad_fn(data, label)
            optimizer(grads)
            epoch_loss += loss.asnumpy()
            step += 1
        avg_loss = epoch_loss / step if step > 0 else 0
        train_losses.append(avg_loss)

        net.set_train(False)
        correct = 0
        total = 0
        for data, label in val_dataset.create_tuple_iterator():
            if len(data.shape) == 1:
                data = data.reshape(1, -1)
            logits = net(data)
            predict = ms.ops.Argmax(output_type=ms.int32)(logits)
            correct += (predict == label).asnumpy().sum()
            total += len(label)
        val_acc = correct / total if total > 0 else 0
        val_accuracies.append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == num_epochs - 1:
            print(f"Epoch {epoch+1:3d}/{num_epochs}: 训练损失={avg_loss:.4f}, 验证准确率={val_acc:.4f}")

    # 训练集准确率（训练结束后评估）
    net.set_train(False)
    correct = 0
    total = 0
    for data, label in train_dataset.create_tuple_iterator():
        if len(data.shape) == 1:
            data = data.reshape(1, -1)
        logits = net(data)
        predict = ms.ops.Argmax(output_type=ms.int32)(logits)
        correct += (predict == label).asnumpy().sum()
        total += len(label)
    train_acc = correct / total if total > 0 else 0

    print(f"\n训练完成! 最佳验证准确率: {best_val_acc:.4f}, 训练集准确率: {train_acc:.4f}")
    return train_losses, val_accuracies, best_val_acc, train_acc


In [10]:
# 第五部分：主程序 - PCA降维 + MLP 经典对照组
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print("PCA降维 + MLP 经典对照组")
    print("=" * 60)

    # 1. 加载数据
    print("\n步骤1：加载MNIST数据")
    all_images, all_labels = load_mnist_data()
    print(f"数据大小: {len(all_images)}")

    # 2. 划分数据集（与量子实验一致）
    print("\n步骤2：划分训练集和测试集")
    train_images, test_images, train_labels, test_labels = train_test_split(
        all_images, all_labels, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
    print(f"训练集: {len(train_images)} 样本, 测试集: {len(test_images)} 样本")

    # 3. 降维预处理
    print("\n步骤3：PCA降维预处理")
    train_features, pca, scaler = preprocess_data(train_images, train_labels, is_train=True)
    test_features, _, _ = preprocess_data(test_images, test_labels, pca=pca, scaler=scaler, is_train=False)
    print(f"特征形状 - 训练集: {train_features.shape}, 测试集: {test_features.shape}")

    # 4. 构造 MindSpore 数据集
    train_dataset = ds.NumpySlicesDataset(
        data=(train_features.astype(np.float32), train_labels.astype(np.int32)),
        column_names=["features", "labels"], shuffle=True
    ).batch(BATCH_SIZE)
    test_dataset = ds.NumpySlicesDataset(
        data=(test_features.astype(np.float32), test_labels.astype(np.int32)),
        column_names=["features", "labels"], shuffle=False
    ).batch(BATCH_SIZE)

    # 5. 训练 MLP
    train_losses, val_accuracies, best_val_acc, train_acc = train_mlp(train_dataset, test_dataset)

    # 6. 结果汇总
    print("\n" + "=" * 60)
    print("PCA降维 + MLP 经典对照组 结果汇总")
    print("=" * 60)
    print(f"MLP 结构: {PCA_COMPONENTS} -> {MLP_HIDDEN_DIMS[0]} -> {MLP_HIDDEN_DIMS[1]} -> 10")
    print(f"训练集准确率: {train_acc:.4f}")
    print(f"测试集(留出集)最佳准确率: {best_val_acc:.4f}")

    # 7. 保存结果
    np.savez('pca_mlp_control_results.npz',
             hidden_dims=MLP_HIDDEN_DIMS,
             train_acc=train_acc,
             best_val_acc=best_val_acc,
             train_losses=train_losses,
             val_accuracies=val_accuracies)
    print("\n结果已保存到 pca_mlp_control_results.npz")


PCA降维 + MLP 经典对照组

步骤1：加载MNIST数据
正在加载MNIST数据集...
总样本数: 70000, 采样后: 70000
类别分布: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64), array([6903, 7877, 6990, 7141, 6824, 6313, 6876, 7293, 6825, 6958],
      dtype=int64))
数据大小: 70000

步骤2：划分训练集和测试集
训练集: 56000 样本, 测试集: 14000 样本

步骤3：PCA降维预处理
PCA解释方差比: 0.4384
特征形状 - 训练集: (56000, 8), 测试集: (14000, 8)

开始训练 MLP (结构: 8 -> 64 -> 32 -> 10)

开始训练循环 (150 epochs)...
Epoch   1/150: 训练损失=0.6413, 验证准确率=0.8648
Epoch  10/150: 训练损失=0.3448, 验证准确率=0.8840
Epoch  20/150: 训练损失=0.3346, 验证准确率=0.8847
Epoch  30/150: 训练损失=0.3304, 验证准确率=0.8855
Epoch  40/150: 训练损失=0.3293, 验证准确率=0.8862
Epoch  50/150: 训练损失=0.3274, 验证准确率=0.8863
Epoch  60/150: 训练损失=0.3264, 验证准确率=0.8856
Epoch  70/150: 训练损失=0.3248, 验证准确率=0.8873
Epoch  80/150: 训练损失=0.3239, 验证准确率=0.8887
Epoch  90/150: 训练损失=0.3238, 验证准确率=0.8873
Epoch 100/150: 训练损失=0.3227, 验证准确率=0.8886
Epoch 110/150: 训练损失=0.3213, 验证准确率=0.8869
Epoch 120/150: 训练损失=0.3203, 验证准确率=0.8873
Epoch 130/150: 训练损失=0.3201, 验证准确率=0.8871
Epoch 140/150: 训练损失